# Scientific figure generation

**REDLINC XII | Computational workflow | Python**

Generate publication-style figures from locally produced analytical summaries.

> **Code-only notebook.** This file contains no embedded execution outputs. It requires locally configured inputs and has not been validated end-to-end in a clean environment.


## Before running

1. Install the packages described in the repository README.
2. Configure local input and output locations without committing any local files.
3. Review variable coding, model assumptions and execution order.
4. Keep generated outputs outside version control.


## Visualization block 1

The following code block is preserved from the reviewed draft. Read its internal comments for detailed operations and verify all required inputs before execution.


In [ ]:
# =============================================================================
# SEM V7 — FIGURES 1 & 2  |  V6
# V6 changes (Fig 2 only):
#   1. NS heatmap cells: no text shown (was gray β value)
#   2. Heatmap columns tighter: gap 0.015 → 0.004
#   3. Indirect cells use continuous α-gradient based on |β| magnitude
#      (significant = full color; ns = same hue but very faint)
#   4. Forest axhspan: boundaries recomputed from actual reversed row indices
# =============================================================================

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
from matplotlib.lines import Line2D
import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "font.family":      "sans-serif",
    "font.sans-serif":  ["Arial","Helvetica","DejaVu Sans"],
    "font.size":        12,
    "savefig.dpi":      300,
    "savefig.bbox":     "tight",
    "savefig.facecolor":"white",
})

C = {
    "ow":      "#F09536",
    "ow_dark": "#E55127",
    "ow_bg":   "#FFF3E0",
    "ow_bd":   "#E65100",
    "ob":      "#A754BE",
    "ob_dark": "#691B99",
    "ob_bg":   "#EDE7F6",
    "ob_bd":   "#5E2CA5",
    "cell_ow_sig":  "#F09536",
    "cell_ow_ns":   "#FFFFFF",
    "cell_ob_sig":  "#A754BE",   # V6: same hue as OB (was very light cream)
    "cell_ob_ns":   "#FFFFFF",
    "cell_ow_tot":  "#F09536",
    "cell_ow_tot2": "#E55127",
    "cell_ob_tot":  "#A754BE",
    "cell_ob_tot2": "#691B99",
    "pos":     "#2E7D32",
    "neg":     "#C62828",
    "ns":      "#9E9E9E",
    "ref":     "#37474F",
    "node":    "#F0F4F8",
    "node_bd": "#455A64",
    "cov":     "#ECEFF1",
    "cov_bd":  "#B0BEC5",
    "moca_bg": "#E8F5E9",
    "text":    "#1C2B3A",
}
DPI = 300

def sig_stars(p, q=None):
    v = q if (q is not None and not pd.isna(q)) else p
    if pd.isna(v):   return ""
    if v < 0.001:    return "****"
    if v < 0.010:    return "***"
    if v < 0.050:    return "**"
    if v < 0.100:    return "†"
    return "ns"

def arrow_col(b, sig):
    if not sig: return C["ns"]
    return C["pos"] if b > 0 else C["neg"]

def arrow_lw(b, mx=3.5, mn=0.8):
    return mn + (mx-mn)*min(abs(b)/0.25, 1.0)

def midpoint_with_offset(x0,y0,x1,y1, perp_dist=0.3, along=0.5):
    mx = x0 + along*(x1-x0)
    my = y0 + along*(y1-y0)
    dx = x1-x0; dy = y1-y0
    length = max(np.sqrt(dx**2+dy**2), 1e-6)
    px = -dy/length; py = dx/length
    return mx + px*perp_dist, my + py*perp_dist

# ── Load ──────────────────────────────────────────────────────────────────────
reg  = pd.read_csv("outputs/SEM_V7_Regresiones.csv")
ind  = pd.read_csv("outputs/SEM_V7_Efectos_Indirectos.csv")
fidx = pd.read_csv("outputs/SEM_V7_Indices_Ajuste_R2.csv")

ind  = ind[ind["BMI_Group"] != "Underweight vs Normal"].copy()
fi   = {r["Metric"]: r["Value"] for _, r in fidx.iterrows()}
reg_lu = {(r["lhs"], r["rhs"]): r for _, r in reg.iterrows()}

def get_reg(lhs, rhs):
    r = reg_lu.get((lhs, rhs))
    if r is None: return None, None, None, False
    p = float(r["pvalue"])
    q = float(r.get("q_val_bmi", r.get("q_val_global", p)))
    return float(r["std.all"]), p, q, p < 0.05

pathway_labels = {
    "Sleep_MRS_MoCA":         "Sleep→MRS→MoCA",
    "Sleep_MRS_SARCF_MoCA":   "Sleep→MRS→SARC-F→MoCA",
    "MRS_MoCA":               "MRS→MoCA",
    "MRS_SARCF_MoCA":         "MRS→SARC-F→MoCA",
    "SARCF_MoCA":             "SARC-F→MoCA",
    "Total_Indirect":         "Total Indirect→MoCA",
    "Total_Effect":           "Total Effect (Direct+Indirect)",
    "Surgery_SARCF_MoCA":     "Surgery→SARC-F→MoCA",
    "Surgery_MRS_MoCA":       "Surgery→MRS→MoCA",
    "Surgery_MRS_SARCF_MoCA": "Surgery→MRS→SARC-F→MoCA",
    "THM_Sleep_MRS_MoCA":     "MHT→Sleep→MRS→MoCA",
}

group_colors = {
    "Overweight vs Normal": C["ow"],
    "Obese vs Normal":      C["ob"],
    "Reference (non-BMI)":  C["ref"],
}

# =============================================================================
# FIGURE 1 — PATH DIAGRAM  (unchanged from V5)
# =============================================================================
print("Generating Figure 1...")

fig, ax = plt.subplots(figsize=(17, 9.5))
ax.set_xlim(0, 17); ax.set_ylim(0, 10); ax.axis("off")

N = {
    "OW":(5.0,9.2),    "OB":(8.5,9.2),
    "Sleep":(6.7,7.2), "MRS":(6.7,5.0),
    "SARCF":(11.5,5.0),"MoCA":(11.5,2.8),
    "MHT":(1.3,8.4),   "Phys":(1.3,7.2),
    "Gyn":(1.3,6.0),   "Age":(1.3,4.8), "Edu":(1.3,3.6),
    "Hyp":(15.5,7.5),  "Dia":(15.5,6.3),
    "Dys":(15.5,5.1),  "Meno":(15.5,3.9),
}
NW,NH=2.1,0.65; CW,CH=1.4,0.46; SW,SH=1.5,0.42

def box(pos, lbl, w=NW, h=NH, bg=C["node"], bd=C["node_bd"],
        fs=12, bold=False, bw=1.3):
    x,y=pos
    ax.add_patch(FancyBboxPatch((x-w/2,y-h/2),w,h,
        boxstyle="round,pad=0.06",facecolor=bg,edgecolor=bd,
        linewidth=bw,zorder=3))
    ax.text(x,y,lbl,ha="center",va="center",fontsize=fs,
            color=C["text"],fontweight="bold" if bold else "normal",zorder=4)

box(N["OW"],   "BMI\nOverweight",           bg=C["ow_bg"],  bd=C["ow_bd"],  bold=True,bw=2.0)
box(N["OB"],   "BMI\nObese",                bg=C["ob_bg"],  bd=C["ob_bd"],  bold=True,bw=2.0)
box(N["Sleep"],"Sleep Quality\n(Jenkins Latent)")
box(N["MRS"],  "Menopausal Symptoms\n(MRS Latent)")
box(N["SARCF"],"SARC-F")
box(N["MoCA"], "Cognition\n(MoCA)",bg=C["moca_bg"],bd=C["pos"],bold=True,bw=1.8)
for k,l in [("MHT","MHT"),("Phys","Physical\nActivity"),
             ("Gyn","Gynecol.\nSurgery"),("Age","Age"),("Edu","Education")]:
    box(N[k],l,w=CW,h=CH,bg=C["cov"],bd=C["cov_bd"],fs=12)
for k,l in [("Hyp","Hypertension"),("Dia","Diabetes"),
             ("Dys","Dyslipidemia"),("Meno","Time since\nMenopause")]:
    box(N[k],l,w=SW,h=SH,bg=C["cov"],bd=C["cov_bd"],fs=12)

def get_ep(cx,cy,edge,w,h):
    if edge=="bottom": return cx, cy-h/2-0.04
    if edge=="top":    return cx, cy+h/2+0.04
    if edge=="left":   return cx-w/2-0.04, cy
    if edge=="right":  return cx+w/2+0.04, cy
    return cx,cy

def draw_arrow_labeled(src_pos, dst_pos, col, lw, ls, rad,
                        src_w, src_h, dst_w, dst_h,
                        se, de, beta=None, p=None, sig=False,
                        label=True, alpha=1.0,
                        perp=0.35, along=0.45, fs=12):
    x0,y0=src_pos; x1,y1=dst_pos
    ep0 = get_ep(x0,y0,se,src_w,src_h)
    ep1 = get_ep(x1,y1,de,dst_w,dst_h)
    ax.annotate("", xy=ep1, xytext=ep0,
        arrowprops=dict(arrowstyle="-|>",color=col,lw=lw,
                        linestyle=ls,
                        connectionstyle=f"arc3,rad={rad}"),
        zorder=2, alpha=alpha)
    if label and beta is not None and sig:
        lx, ly = midpoint_with_offset(ep0[0],ep0[1],ep1[0],ep1[1],
                                       perp_dist=perp, along=along)
        stars = sig_stars(p)
        txt   = f"β={'+'if beta>=0 else ''}{beta:.3f}{stars}"
        ax.text(lx, ly, txt, ha="center", va="center", fontsize=fs,
                color=col, fontweight="bold",
                bbox=dict(facecolor="white",edgecolor="none",
                          alpha=0.92,pad=1.1,boxstyle="round,pad=0.15"),
                zorder=7)

cov_map = {
    "MHT": ("THMTotal",[("Jenkins_Latent","Sleep","right","left",0.0),
                         ("MRS_Latent",    "MRS",  "right","left",0.05)]),
    "Phys":("ActFisica",[("MRS_Latent",   "MRS",  "right","left",0.0),
                          ("SARCF_Total",  "SARCF","right","left",0.0)]),
    "Gyn": ("GynSurgery",[("Jenkins_Latent","Sleep","right","left",0.0),
                           ("MRS_Latent",   "MRS",  "right","left",-0.05),
                           ("SARCF_Total",  "SARCF","right","left",0.05),
                           ("MOCA_OUT",     "MoCA", "right","left",0.0)]),
    "Age": ("Edad",[("Jenkins_Latent","Sleep","right","left",0.05),
                    ("SARCF_Total",   "SARCF","right","left",-0.05),
                    ("MOCA_OUT",      "MoCA", "right","left",0.0)]),
    "Edu": ("Estudio",[("MOCA_OUT","MoCA","right","left",0.0)]),
    "Hyp": ("HipertArt",[("Jenkins_Latent","Sleep","left","right",0.0),
                          ("MRS_Latent",    "MRS",  "left","right",-0.05),
                          ("SARCF_Total",   "SARCF","left","right",0.0),
                          ("MOCA_OUT",      "MoCA", "left","right",0.05)]),
    "Dia": ("Diabete",[("Jenkins_Latent","Sleep","left","right",0.05),
                        ("MRS_Latent",    "MRS",  "left","right",0.0),
                        ("SARCF_Total",   "SARCF","left","right",-0.05),
                        ("MOCA_OUT",      "MoCA", "left","right",0.0)]),
    "Dys": ("Colesterol",[("Jenkins_Latent","Sleep","left","right",0.0),
                           ("MRS_Latent",    "MRS",  "left","right",0.05),
                           ("SARCF_Total",   "SARCF","left","right",0.05),
                           ("MOCA_OUT",      "MoCA", "left","right",-0.05)]),
    "Meno":("MenoYears",[("Jenkins_Latent","Sleep","left","right",-0.05),
                          ("MRS_Latent",    "MRS",  "left","right",0.0),
                          ("SARCF_Total",   "SARCF","left","right",-0.05),
                          ("MOCA_OUT",      "MoCA", "left","right",0.0)]),
}
left_cov={"MHT","Phys","Gyn","Age","Edu"}
for ck,(rhs,cons) in cov_map.items():
    is_l=ck in left_cov
    cw_=CW if is_l else SW; ch_=CH if is_l else SH
    for (lc,dk,se,de,rad) in cons:
        beta,p,q,sig=get_reg(lc,rhs)
        col=arrow_col(beta,sig) if (beta is not None and sig) else C["ns"]
        lw_=(arrow_lw(beta) if sig else 0.55) if beta is not None else 0.55
        draw_arrow_labeled(N[ck],N[dk],col=col,lw=lw_,ls="-",rad=rad,
                           src_w=cw_,src_h=ch_,dst_w=NW,dst_h=NH,
                           se=se,de=de,label=False,alpha=0.5)

bmi_arrows = [
    ("IMC_Overweight","Jenkins_Latent","Sleep",-0.3, 0.06, 0.30, 0.50,"bottom","top"),
    ("IMC_Obese",     "Jenkins_Latent","Sleep", 0.3,-0.06,-0.30, 0.50,"bottom","top"),
    ("IMC_Overweight","MRS_Latent",    "MRS",  -0.5, 0.10, 0.30, 0.45,"bottom","top"),
    ("IMC_Obese",     "MRS_Latent",    "MRS",   0.5,-0.10,-0.30, 0.45,"bottom","top"),
    ("IMC_Overweight","SARCF_Total",   "SARCF", 0.1,-0.20,-0.30, 0.50,"bottom","top"),
    ("IMC_Obese",     "SARCF_Total",   "SARCF", 0.4,-0.12, 0.30, 0.50,"bottom","top"),
    ("IMC_Overweight","MOCA_OUT",      "MoCA",  0.3,-0.25,-0.30, 0.50,"bottom","top"),
    ("IMC_Obese",     "MOCA_OUT",      "MoCA",  0.6,-0.15, 0.30, 0.50,"bottom","top"),
]
bmi_style={"IMC_Overweight":(C["ow"],"--"),"IMC_Obese":(C["ob"],":")}

for (rhs,lhs,dk,sx,rad,perp,along,se,de) in bmi_arrows:
    beta,p,q,sig=get_reg(lhs,rhs)
    if beta is None: continue
    col,ls=bmi_style[rhs]
    is_ow="Over" in rhs
    src=(N["OW"][0]+sx if is_ow else N["OB"][0]+sx,
         N["OW"][1]    if is_ow else N["OB"][1])
    lw_=arrow_lw(beta) if sig else 0.7
    col_draw=col if sig else C["ns"]
    draw_arrow_labeled(src,N[dk],col=col_draw,lw=lw_,ls=ls,rad=rad,
                       src_w=0.1,src_h=0.1,dst_w=NW,dst_h=NH,
                       se=se,de=de,
                       beta=beta,p=p,sig=sig,label=sig,
                       alpha=1.0 if sig else 0.45,
                       perp=perp,along=along,fs=12)

cascade=[
    ("MRS_Latent", "Jenkins_Latent","Sleep","MRS",  0.0, 0.35, 0.50,"bottom","top"),
    ("SARCF_Total","MRS_Latent",    "MRS",  "SARCF",0.0,-0.25, 0.50,"right", "left"),
    ("MOCA_OUT",   "MRS_Latent",    "MRS",  "MoCA",-0.2,-0.35, 0.50,"bottom","top"),
    ("MOCA_OUT",   "SARCF_Total",   "SARCF","MoCA", 0.0, 0.30, 0.50,"bottom","top"),
]
for (lhs,rhs,sk,dk,rad,perp,along,se,de) in cascade:
    beta,p,q,sig=get_reg(lhs,rhs)
    if beta is None: continue
    col=arrow_col(beta,sig); lw_=arrow_lw(beta) if sig else 0.8
    draw_arrow_labeled(N[sk],N[dk],col=col,lw=lw_,ls="-",rad=rad,
                       src_w=NW,src_h=NH,dst_w=NW,dst_h=NH,
                       se=se,de=de,
                       beta=beta,p=p,sig=sig,label=True,
                       perp=perp,along=along,fs=12)

legend_elems=[
    Line2D([0],[0],color=C["ow"], lw=2,ls="--",label="Overweight vs Normal (sig.)"),
    Line2D([0],[0],color=C["ns"], lw=1,ls="--",label="Overweight vs Normal (ns)"),
    Line2D([0],[0],color=C["ob"], lw=2,ls=":", label="Obese vs Normal (sig.)"),
    Line2D([0],[0],color=C["ns"], lw=1,ls=":", label="Obese vs Normal (ns)"),
    Line2D([0],[0],color=C["pos"],lw=2.5,      label="Positive structural path"),
    Line2D([0],[0],color=C["neg"],lw=2.5,      label="Negative structural path"),
    Line2D([0],[0],color=C["ns"], lw=1.0,      label="Covariate path (unlabelled)"),
]
ax.legend(handles=legend_elems,loc="upper right",bbox_to_anchor=(1.0,1.0),
          fontsize=12,framealpha=0.95,edgecolor="#B0BEC5",
          handlelength=2.2,labelspacing=0.35,borderpad=0.6,
          title="Legend",title_fontsize=12)

cfi=fi.get("cfi",np.nan); tli=fi.get("tli",np.nan)
rmsea=fi.get("rmsea",np.nan); srmr=fi.get("srmr",np.nan)
note=(
    "Note: Only significant BMI paths labelled. Covariate paths shown without β.  "
    "MHT→Sleep,MRS  |  Physical Activity→MRS,SARC-F  |  Gynecol. Surgery→Sleep,MRS,SARC-F,MoCA\n"
    "Age→Sleep,SARC-F,MoCA  |  Education→MoCA  |  "
    "Hypertension/Diabetes/Dyslipidemia/Time since Menopause→Sleep,MRS,SARC-F,MoCA  |  "
    "Sleep affects MoCA via MRS only (no direct path by design).\n"
    f"†p<.10  *p<.05  **p<.01  ***p<.001  ****p<.0001    "
    f"Model fit (N=1,165): CFI={cfi:.3f}  TLI={tli:.3f}  RMSEA={rmsea:.3f}  SRMR={srmr:.3f}"
)
ax.text(0.01,0.01,note,transform=ax.transAxes,fontsize=10.0,color="#546E7A",
        va="bottom",ha="left",style="italic",
        bbox=dict(facecolor="#F8F9FA",edgecolor="#B0BEC5",
                  alpha=0.92,pad=4,boxstyle="round,pad=0.3"))

plt.tight_layout(pad=0.3)
plt.savefig("Fig1_PathDiagram_V7.png",dpi=DPI)
plt.show()
print("→ Fig1_PathDiagram_V7.png saved.\n")


# =============================================================================
# FIGURE 2 — Heatmap + Forest Plot  (V6 fixes)
# =============================================================================
print("Generating Figure 2...")

direct_order =["Jenkins_Latent","MRS_Latent","SARCF_Total","MOCA_OUT"]
direct_labels={"Jenkins_Latent":"Sleep","MRS_Latent":"MRS",
               "SARCF_Total":"SARC-F","MOCA_OUT":"MoCA"}

direct_rows=[]
for node in direct_order:
    for grp,rhs in [("Overweight vs Normal","IMC_Overweight"),
                     ("Obese vs Normal",     "IMC_Obese")]:
        beta,p,q,sig=get_reg(node,rhs)
        if beta is None: continue
        direct_rows.append({"Group":grp,"Effect":direct_labels[node],
                             "B":beta,"p":p,"q_val":q,"Sig":sig,
                             "Section":"DIRECT"})

pathway_order_bmi_heat  =["Sleep_MRS_MoCA","Sleep_MRS_SARCF_MoCA","MRS_MoCA",
                           "MRS_SARCF_MoCA","SARCF_MoCA",
                           "Total_Indirect","Total_Effect"]
pathway_order_bmi_forest=["Sleep_MRS_MoCA","Sleep_MRS_SARCF_MoCA","MRS_MoCA",
                           "MRS_SARCF_MoCA","SARCF_MoCA","Total_Indirect"]
pathway_order_ref       =["Surgery_SARCF_MoCA","Surgery_MRS_MoCA",
                           "Surgery_MRS_SARCF_MoCA","THM_Sleep_MRS_MoCA"]

ind_rows=[]
for _,r in ind.iterrows():
    pth=r["Pathway"]; grp=r["BMI_Group"]
    if pth not in pathway_labels: continue
    is_total=pth in("Total_Indirect","Total_Effect")
    ind_rows.append({
        "Group":grp,"Effect":pathway_labels[pth],"Pathway":pth,
        "B":r["B"],"SE":r["SE"],"p":r["p"],
        "q_val":r.get("q_val",r["p"]),
        "CI_low":r["CI_low"],"CI_high":r["CI_high"],
        "Sig":bool(r["Significant"]),
        "Section":"TOTAL" if is_total else "INDIRECT",
    })

fig=plt.figure(figsize=(18,9))
gs=fig.add_gridspec(1,2,width_ratios=[1.0,1.3],
                    left=0.01,right=0.99,
                    bottom=0.06,top=0.96,wspace=0.30)
axA=fig.add_subplot(gs[0])
axB=fig.add_subplot(gs[1])
fig.text(0.01,0.98,"A",fontsize=16,fontweight="bold",color=C["text"])
fig.text(0.52,0.98,"B",fontsize=16,fontweight="bold",color=C["text"])
axA.axis("off")

# ── PANEL A — HEATMAP ────────────────────────────────────────────────────────
sections=[
    ("DIRECT",  [r for r in direct_rows]),
    ("INDIRECT",[r for r in ind_rows
                 if r["Section"]=="INDIRECT" and r["Group"]!="Reference (non-BMI)"]),
    ("TOTAL",   [r for r in ind_rows
                 if r["Section"]=="TOTAL"    and r["Group"]!="Reference (non-BMI)"]),
]

row_labels=[]
seen=set()
for sec,rows in sections:
    for r in rows:
        if r["Group"]=="Overweight vs Normal" and r["Effect"] not in seen:
            row_labels.append((sec,r["Effect"])); seen.add(r["Effect"])

col_keys=["Overweight vs Normal","Obese vs Normal"]
nrows=len(row_labels)

cell_h=0.76/nrows
cell_w=0.32
# FIX 2: tighter column gap (0.015 → 0.004)
x0s=[0.27, 0.27+cell_w+0.004]
y_top=0.88

mat_b  =pd.DataFrame(index=[l for _,l in row_labels],columns=col_keys,dtype=float)
mat_p  =pd.DataFrame(index=[l for _,l in row_labels],columns=col_keys,dtype=float)
mat_q  =pd.DataFrame(index=[l for _,l in row_labels],columns=col_keys,dtype=float)
mat_sig=pd.DataFrame(index=[l for _,l in row_labels],columns=col_keys,dtype=bool)

for r in direct_rows+ind_rows:
    eff=r["Effect"]; grp=r["Group"]
    if eff in mat_b.index and grp in col_keys:
        mat_b.loc[eff,grp]=r["B"]; mat_p.loc[eff,grp]=r["p"]
        mat_q.loc[eff,grp]=r["q_val"]; mat_sig.loc[eff,grp]=r["Sig"]

# FIX 3: compute max |β| per section for gradient scaling
# Collect all indirect β values (excluding totals and direct)
indirect_betas = []
for sec,eff in row_labels:
    if sec == "INDIRECT":
        for grp in col_keys:
            b = mat_b.loc[eff, grp]
            if not pd.isna(b):
                indirect_betas.append(abs(b))
beta_max_indirect = max(indirect_betas) if indirect_betas else 0.1

def beta_to_alpha(b_abs, b_max, sig, min_alpha=0.10, max_alpha=1.0):
    """Map |β| to alpha for gradient coloring. Non-sig gets very low alpha."""
    if b_max == 0:
        return max_alpha if sig else min_alpha
    ratio = min(b_abs / b_max, 1.0)
    if sig:
        return min_alpha + (max_alpha - min_alpha) * ratio
    else:
        # Non-sig: even fainter — range [0.05, 0.18]
        return 0.05 + 0.13 * ratio

import matplotlib.colors as mcolors

def hex_with_alpha_on_white(hex_col, alpha):
    """Blend hex color with white at given alpha → return blended hex."""
    r_,g_,b_ = mcolors.to_rgb(hex_col)
    r2 = alpha*r_ + (1-alpha)*1.0
    g2 = alpha*g_ + (1-alpha)*1.0
    b2 = alpha*b_ + (1-alpha)*1.0
    return (r2, g2, b2)

# Column headers
for j,(hdr,col) in enumerate(zip(
        ["Overweight\nvs Normal","Obese\nvs Normal"],
        [C["ow_dark"],C["ob_dark"]])):
    axA.text(x0s[j]+cell_w/2,y_top+0.04,hdr,
             transform=axA.transAxes,ha="center",va="bottom",
             fontsize=11,fontweight="bold",color=col)

prev_sec=None; sep_positions=[]

for i,(sec,eff) in enumerate(row_labels):
    y_bot=y_top-(i+1)*cell_h
    y_mid=y_top-(i+0.5)*cell_h
    if sec!=prev_sec:
        if prev_sec is not None:
            sep_positions.append(y_bot+cell_h)
        prev_sec=sec

    is_total_ind = eff == pathway_labels["Total_Indirect"]
    is_total_eff = eff == pathway_labels.get("Total_Effect","__none__")
    is_total     = is_total_ind or is_total_eff

    axA.text(0.25,y_mid,eff,transform=axA.transAxes,
             ha="right",va="center",
             fontsize=11 if not is_total else 11,
             fontweight="bold" if is_total else "normal",
             color=C["text"])

    for j,grp in enumerate(col_keys):
        xL=x0s[j]
        b=mat_b.loc[eff,grp]; p=mat_p.loc[eff,grp]
        q=mat_q.loc[eff,grp]; sig=mat_sig.loc[eff,grp]
        is_ow=(grp=="Overweight vs Normal")
        base_col = C["ow"] if is_ow else C["ob"]

        if pd.isna(b):
            bg="#F5F5F5"; txt="—"; tc=C["text"]
        else:
            stars=sig_stars(p,q)
            sign="+" if b>=0 else ""
            txt=f"{sign}{b:.3f}"
            if stars not in ("ns", ""):
                txt += stars

            if is_total_eff:
                bg=C["cell_ow_tot2"] if is_ow else C["cell_ob_tot2"]
                tc="white"
            elif is_total_ind:
                bg=C["cell_ow_tot"] if is_ow else C["cell_ob_tot"]
                tc="white"
            elif sec == "INDIRECT":
                # FIX 3: continuous gradient based on |β|
                alph = beta_to_alpha(abs(b), beta_max_indirect, sig)
                bg = hex_with_alpha_on_white(base_col, alph)
                # Text: dark for light cells (ns/small), white for strong sig
                tc = C["text"] if alph < 0.55 else "white"
                if not sig:
                    # FIX 1: ns cells — show β lightly or omit stars, keep value dim
                    tc = "#AAAAAA"
            elif sec == "DIRECT":
                if sig:
                    bg = base_col
                    tc = "white"
                else:
                    bg = "#FFFFFF"
                    # FIX 1: no gray text for ns direct cells
                    tc = "#CCCCCC"
            else:
                bg="#FFFFFF"; tc="#CCCCCC"

        rect=plt.Rectangle((xL,y_bot),cell_w,cell_h,
                            transform=axA.transAxes,
                            facecolor=bg,edgecolor="#CFD8DC",
                            linewidth=0.5,clip_on=False,zorder=2)
        axA.add_patch(rect)

        # FIX 1: for ns cells (non-significant, non-total), show value only faintly
        # (text color already set to #CCCCCC or #AAAAAA above)
        axA.text(xL+cell_w/2,y_mid,txt,transform=axA.transAxes,
                 ha="center",va="center",fontsize=12,
                 fontweight="bold" if (sig or is_total) else "normal",
                 color=tc)

# Section separators
for ysp in sep_positions:
    axA.plot([0.0,1.0],[ysp,ysp],color="#90A4AE",lw=1.2,ls="--",
             transform=axA.transAxes,zorder=5,clip_on=False)

axA.text(0.50,y_top+0.08,"Direct and Indirect BMI Effects on Cognition",
         transform=axA.transAxes,ha="center",va="bottom",
         fontsize=11,fontweight="bold",color=C["text"])
#axA.text(0.50,y_top+0.04,
#         "†q<.10  **q<.05  ***q<.01  ****q<.001",
#         transform=axA.transAxes,ha="center",va="bottom",
#         fontsize=10,color=C["ns"])

# ── PANEL B — FOREST PLOT ────────────────────────────────────────────────────
forest_rows=[]
for grp in ["Overweight vs Normal","Obese vs Normal"]:
    sub=[r for r in ind_rows if r["Group"]==grp]
    for pth in pathway_order_bmi_forest:
        r=next((x for x in sub if x["Pathway"]==pth),None)
        if r is None: continue
        forest_rows.append({
            "label":  pathway_labels.get(pth,""),
            "B":      r["B"],"CI_low":r["CI_low"],"CI_high":r["CI_high"],
            "p":      r["p"],"q_val": r.get("q_val",r["p"]),
            "Sig":    r["Sig"],"color":group_colors[grp],
            "total":  pth=="Total_Indirect","group":grp,
        })
    # Spacer AFTER all pathways of this group
    forest_rows.append({"label":"","B":np.nan,"CI_low":np.nan,
                         "CI_high":np.nan,"p":np.nan,"q_val":np.nan,
                         "Sig":False,"color":"white","total":False,"group":grp})

sub_ref=[r for r in ind_rows if r["Group"]=="Reference (non-BMI)"]
for pth in pathway_order_ref:
    r=next((x for x in sub_ref if x["Pathway"]==pth),None)
    if r is None: continue
    forest_rows.append({
        "label":  pathway_labels.get(pth,pth),
        "B":      r["B"],"CI_low":r["CI_low"],"CI_high":r["CI_high"],
        "p":      r["p"],"q_val": r.get("q_val",r["p"]),
        "Sig":    r["Sig"],"color":C["ref"],
        "total":  False,"group":"Reference (non-BMI)",
    })

plot_df=pd.DataFrame(forest_rows).iloc[::-1].reset_index(drop=True)

axB.set_ylim(-0.8, len(plot_df)-0.2)

# FIX 4: Recompute axhspan boundaries from actual reversed row indices
# After reversal, groups appear in opposite order.
# Build group→[row_indices] map from plot_df
group_row_indices = {}
for idx, row in plot_df.iterrows():
    g = row["group"]
    if g not in group_row_indices:
        group_row_indices[g] = []
    group_row_indices[g].append(idx)

for grp, col in group_colors.items():
    if grp not in group_row_indices:
        continue
    idxs = group_row_indices[grp]
    # Exclude spacer rows (NaN B) from the visible span
    # But include them in the shaded area — use min/max of all rows in group
    y_lo = min(idxs) - 0.5
    y_hi = max(idxs) + 0.5
    axB.axhspan(y_lo, y_hi, alpha=0.06, color=col, zorder=0)

for i,row in plot_df.iterrows():
    if pd.isna(row["B"]): continue
    col=row["color"]; sig=row["Sig"]; total=row["total"]
    alpha=1.0 if sig else 0.55
    lw_=2.5 if total else 1.5; ms=11 if total else 8

    axB.plot([row["CI_low"],row["CI_high"]],[i,i],
             color=col,lw=lw_,alpha=alpha,solid_capstyle="round",zorder=2)
    axB.plot(row["B"],i,
             marker="D" if total else "o",ms=ms,
             color=col if sig else "white",
             markeredgecolor=col,
             markeredgewidth=2.0 if sig else 0.8,
             alpha=alpha,zorder=3)
    stars=sig_stars(row["p"],row["q_val"])
    txt_star=" ns" if (not stars or stars=="ns") else f" {stars}"
    col_star=C["ns"] if txt_star==" ns" else col
    axB.text(row["CI_high"]+0.01,i,txt_star,va="center",
             fontsize=10,color=col_star,
             fontweight="bold" if txt_star!=" ns" else "normal")

axB.axvline(0,color=C["text"],lw=0.8,ls="--",alpha=0.5,zorder=1)
axB.set_yticks(range(len(plot_df)))
axB.set_yticklabels(plot_df["label"],fontsize=10.5)
axB.tick_params(left=False)
axB.set_xlabel("Standardized Indirect Effect (β)",fontsize=11)
axB.spines["top"].set_visible(False)
axB.spines["right"].set_visible(False)

legend_elems=[
    Line2D([0],[0],color=C["ow"],lw=2,marker="o",ms=8,
           markerfacecolor=C["ow"],label="Overweight"),
    Line2D([0],[0],color=C["ob"],lw=2,marker="o",ms=8,
           markerfacecolor=C["ob"],label="Obese"),
    Line2D([0],[0],color=C["ref"],lw=2,marker="o",ms=8,
           markerfacecolor=C["ref"],label="Reference"),
    Line2D([0],[0],color="grey",lw=0,marker="D",ms=9,
           markerfacecolor="grey",label="Total indirect"),
    Line2D([0],[0],color="grey",lw=0,marker="o",ms=7,
           markerfacecolor="white",markeredgecolor="grey",
           label="Non-sig / trend"),
]
axB.legend(handles=legend_elems,loc="lower left",
           fontsize=10,framealpha=0.96,edgecolor="#B0BEC5",
           labelspacing=0.5,borderpad=0.7)

plt.savefig("Fig2_HeatmapForest_V7.png",dpi=DPI)
plt.show()
print("→ Fig2_HeatmapForest_V7.png saved.\n")
print("DONE — Fig1_PathDiagram_V7.png | Fig2_HeatmapForest_V7.png")

## Reproducibility and interpretation

Check execution logs, convergence and assumptions before interpreting any estimates. This repository intentionally excludes study-specific outputs. For scientific interpretation, refer to the associated publication when available.
